## Setup

In [1]:
# Get raw advent-of-code data
from aocd.models import Puzzle

puzzle = Puzzle(year=2015, day=18)
input_data = puzzle.input_data
example = puzzle.examples[0]

In [2]:
# Import performance checking utility
from aoc_utils.perf_check import check_example, time_solution

## Part a

### Basic iterative approach
A game-of-life kind of problem. Let's start with a simple iterative approach.

In [3]:
# Imports
from aoc_utils.dict_grid import get_octagonal_neighbors, map_grid_values, text_to_grid_dict

In [ ]:
# Functions
def solve_a_iterative_basic(input_data: str, *, steps: int = 100) -> int:
    """Iterate over all lights for each step, and update the grid based on the rules."""
    # Load the grid, map to 1s and 0s for on and off
    mapping: dict[str, int] = {"#": 1, ".": 0}
    grid = map_grid_values(text_to_grid_dict(input_data), mapping)

    for _ in range(steps):
        new_grid = grid.copy()
        for pos, light in grid.items():
            # Count the on-neighbors
            on_neighbor_count = sum(grid.get(n, 0) for n in get_octagonal_neighbors(pos))

            # Apply the rules
            stay_on = light == 1 and on_neighbor_count in (2, 3)
            turn_on = light == 0 and on_neighbor_count == 3
            new_grid[pos] = int(stay_on or turn_on)

        # Update the grid for the next step
        grid = new_grid

    # Count the number of lights that are on
    return sum(grid.values())

In [5]:
# Correctness check
check_example(solve_a_iterative_basic, example, steps=4)

solve_a_iterative_basic found answer 4, which is the correct solution for part A!


In [6]:
# Performance check
time_a_iterative_basic = time_solution(solve_a_iterative_basic, input_data, iterations=10, runs=3, time_unit="s")

solve_a_iterative_basic takes 0.88 s


### Optimized iterative approach
Instead of fetching the neighbors for each light, we can keep track of the on-neighbor counts in a separate grid, and update it as we go. We can using convolution to do this efficiently.

In [7]:
# Imports
import numpy as np
from scipy.signal import convolve2d

import aoc_utils.numpy_grid as np_grid

In [ ]:
# Functions
def solve_a_iterative_optimized(input_data: str, *, steps: int = 100) -> int:
    """Update the light grid for each step using convolution-based neighbor counts."""
    # Load the grid as binary values (1 for on, 0 for off)
    grid = np_grid.map_grid_values(np_grid.text_to_array_grid(input_data), mapping={"#": 1, ".": 0}).astype(np.int8)

    for _ in range(steps):
        # Count on-neighbors for each position at once using convolution
        neighbor_counts = convolve2d(grid, np_grid.OCTAGONAL_KERNEL, mode="same", boundary="fill")

        # Apply the rules via boolean masks
        stay_on = (grid == 1) & ((neighbor_counts == 2) | (neighbor_counts == 3))
        turn_on = (grid == 0) & (neighbor_counts == 3)
        grid = (stay_on | turn_on).astype(np.int8)

    # Count the number of lights that are on after all steps
    return int(grid.sum())

In [9]:
# Correctness check
check_example(solve_a_iterative_optimized, example, steps=4)

solve_a_iterative_optimized found answer 4, which is the correct solution for part A!


In [11]:
# Performance check
time_a_iterative_optimized = time_solution(solve_a_iterative_optimized, input_data)
print(
    f"Which is {time_a_iterative_basic * 1_000 / time_a_iterative_optimized:.0f} x faster than the dict-based solution!"
)

solve_a_iterative_optimized takes 26.67 ms
Which is 33 x faster than the dict-based solution!


In [12]:
# Submit answer
puzzle.answer_a = solve_a_iterative_optimized(input_data)

That's the right answer!  You are one gold star closer to powering the weather machine. [Continue to Part Two]


## Part b

Same as part a, but now we'll force the corner lights to stay on after each step. I've just added a few lines ot the optimized solution for part a to achieve this.

In [34]:
# Functions
def force_corners(grid: np.ndarray) -> None:
    """Helper function to force the corners of a grid to stay on."""
    grid[0, 0] = 1
    grid[0, -1] = 1
    grid[-1, 0] = 1
    grid[-1, -1] = 1


def solve_b(input_data: str, *, steps: int = 100) -> int:
    """Update the light grid for each step using convolution-based neighbor counts, but force the corners to stay on."""
    # Load the grid as binary values (1 for on, 0 for off)
    grid = np_grid.map_grid_values(np_grid.text_to_array_grid(input_data), mapping={"#": 1, ".": 0}).astype(np.int8)

    # Force the corner lights to stay on
    force_corners(grid)

    for _ in range(steps):
        # Count on-neighbors for each position
        neighbor_counts = convolve2d(grid, np_grid.OCTAGONAL_KERNEL, mode="same", boundary="fill")

        # Apply the rules via boolean masks
        stay_on = (grid == 1) & ((neighbor_counts == 2) | (neighbor_counts == 3))
        turn_on = (grid == 0) & (neighbor_counts == 3)
        grid = (stay_on | turn_on).astype(np.int8)

        # Force the corner lights to stay on
        force_corners(grid)

    # Count the number of lights that are on after all steps
    return int(grid.sum())

In [35]:
# Correctness check (manual check as the example doesn't specify the expected result for part b)
solve_b(example.input_data, steps=5) == 17

True

In [36]:
# Performance check
time_b_iterative = time_solution(solve_b, input_data, iterations=10)

solve_b takes 26.70 ms


In [38]:
# Submit answer
puzzle.answer_b = solve_b(input_data)

That's the right answer!  You are one gold star closer to powering the weather machine.You have completed Day 18! You can [Shareon
  Bluesky
Twitter
Mastodon] this victory or [Return to Your Advent Calendar].
